In [ ]:
# 1.1 SETUP: GOOGLE COLAB AND BIGQUERY CONNECTION

# Authenticate user in Google Colab
from google.colab import auth
auth.authenticate_user()

# Import libraries for data manipulation
import pandas as pd
import numpy as np

# Import libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import statistical functions for A/B testing
from scipy.stats import norm

# Import BigQuery client
from google.cloud import bigquery

# Create BigQuery client
client = bigquery.Client(project="data-analytics-mate")

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Set visualization style
sns.set_theme(style="whitegrid")

In [ ]:
# 1.2 DATA EXTRACTION FROM BIGQUERY

# Define SQL query to aggregate A/B test data by test and test group
query = """
WITH session_info AS (
  SELECT
    s.ga_session_id,
    ab.test,
    ab.test_group
  FROM `DA.ab_test` ab
  JOIN `DA.session` s
    ON ab.ga_session_id = s.ga_session_id
),

events_agg AS (
  SELECT
    si.test,
    si.test_group,
    COUNT(DISTINCT si.ga_session_id) AS session,

    -- Event occurrences are counted instead of distinct sessions
    -- to match the original dashboard calculation logic
    COUNT(CASE
      WHEN ep.event_name = 'add_payment_info' THEN ep.ga_session_id
    END) AS add_payment_info,

    COUNT(CASE
      WHEN ep.event_name = 'add_shipping_info' THEN ep.ga_session_id
    END) AS add_shipping_info,

    COUNT(CASE
      WHEN ep.event_name = 'begin_checkout' THEN ep.ga_session_id
    END) AS begin_checkout

  FROM session_info si
  LEFT JOIN `DA.event_params` ep
    ON si.ga_session_id = ep.ga_session_id
  GROUP BY
    si.test,
    si.test_group
),

accounts_agg AS (
  SELECT
    si.test,
    si.test_group,

    -- Count sessions that created a new account
    COUNT(DISTINCT acs.ga_session_id) AS new_accounts

  FROM session_info si
  LEFT JOIN `DA.account_session` acs
    ON si.ga_session_id = acs.ga_session_id
  GROUP BY
    si.test,
    si.test_group
)

SELECT
  e.test,
  e.test_group,
  e.session,
  e.add_payment_info,
  e.add_shipping_info,
  e.begin_checkout,
  a.new_accounts
FROM events_agg e
LEFT JOIN accounts_agg a
  ON e.test = a.test
  AND e.test_group = a.test_group
ORDER BY
  e.test,
  e.test_group
"""

In [ ]:
# Run SQL query and load results into a pandas DataFrame
# The BigQuery Storage API is disabled to avoid permission issues
df = client.query(query).result().to_dataframe(create_bqstorage_client=False)

# Display the extracted dataset
df

,test,test_group,session,add_payment_info,add_shipping_info,begin_checkout,new_accounts
0,1,1,45362,1988,3034,3784,3823
1,1,2,45193,2229,3221,4021,3681
2,2,1,50637,2344,3480,4262,4165
3,2,2,50244,2409,3510,4313,4184
4,3,1,70047,3623,5298,9532,5856
5,3,2,70439,3697,5188,9264,5822
6,4,1,105079,3731,5128,12555,8984
7,4,2,105141,3601,4956,12267,8687


In [ ]:
# 1.3 DATA VALIDATION

# Check dataset shape
print(f"Dataset shape: {df.shape}")

# Check for missing values
print("\nMissing values:")
print(df.isna().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

Dataset shape: (8, 7)

Missing values:
test                 0
test_group           0
session              0
add_payment_info     0
add_shipping_info    0
begin_checkout       0
new_accounts         0
dtype: int64

Data types:
test                 Int64
test_group           Int64
session              Int64
add_payment_info     Int64
add_shipping_info    Int64
begin_checkout       Int64
new_accounts         Int64
dtype: object


In [ ]:
# 2. METRIC CONFIGURATION

# Define conversion metrics
metrics = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new_accounts"
]

metrics

['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_accounts']

In [ ]:
# 3. STATISTICAL FUNCTIONS

from scipy.stats import norm
import math

def calculate_z_test(success_test, total_test, success_control, total_control):
    """
    Calculate z-statistic and p-value for two-proportion z-test.
    """

    # Conversion rates
    p1 = success_test / total_test
    p2 = success_control / total_control

    # Pooled probability
    pooled_probability = (
        success_test + success_control
    ) / (
        total_test + total_control
    )

    # Standard error
    standard_error = math.sqrt(
        pooled_probability *
        (1 - pooled_probability) *
        ((1 / total_test) + (1 / total_control))
    )

    # Z statistic
    z_stat = (p1 - p2) / standard_error

    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))

    return z_stat, p_value

## Metric Definition

Conversion metrics are calculated as the ratio of total event occurrences to total sessions within each experiment group.

**Formula**

metric = event_count / session_count

**Example**

add_payment_info / session

In [ ]:
# 4. STATISTICAL SIGNIFICANCE CALCULATION

results = []

# Iterate through each A/B test
for test_number in sorted(df["test"].unique()):

    # Split test data into control and variant groups
    control = df[
        (df["test"] == test_number) &
        (df["test_group"] == 1)
    ].iloc[0]

    variant = df[
        (df["test"] == test_number) &
        (df["test_group"] == 2)
    ].iloc[0]

    # Iterate through all conversion metrics
    for metric in metrics:

        numerator_test = variant[metric]
        denominator_test = variant["session"]

        numerator_control = control[metric]
        denominator_control = control["session"]

        # Calculate conversion rates
        conversion_rate_test = numerator_test / denominator_test
        conversion_rate_control = numerator_control / denominator_control

        # Calculate metric uplift (%)
        metric_change = (
            (conversion_rate_test - conversion_rate_control)
            / conversion_rate_control
        ) * 100

        # Calculate statistical significance
        z_stat, p_value = calculate_z_test(
            numerator_test,
            denominator_test,
            numerator_control,
            denominator_control
        )

        # Save results
        results.append({
            "test_number": test_number,
            "metric": metric,
            "numerator_event": numerator_test,
            "denominator": denominator_test,
            "numerator_control": numerator_control,
            "denominator_control": denominator_control,
            "conversion_rate_test": conversion_rate_test,
            "conversion_rate_control": conversion_rate_control,
            "metric_change": metric_change,
            "z_stat": z_stat,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

In [ ]:
# Convert results into DataFrame
results_df = pd.DataFrame(results)

# Display results
results_df

,test_number,metric,numerator_event,denominator,numerator_control,denominator_control,conversion_rate_test,conversion_rate_control,metric_change,z_stat,p_value,significant
0,1,add_payment_info,2229,45193,1988,45362,0.049322,0.043825,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,3221,45193,3034,45362,0.071272,0.066884,6.560481,2.603571,0.009226,True
2,1,begin_checkout,4021,45193,3784,45362,0.088974,0.083418,6.660587,2.978783,0.002894,True
3,1,new_accounts,3681,45193,3823,45362,0.081451,0.084278,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,2409,50244,2344,50637,0.047946,0.046290,3.576911,1.240994,0.214608,False
5,2,add_shipping_info,3510,50244,3480,50637,0.069859,0.068724,1.650995,0.709557,0.477979,False
6,2,begin_checkout,4313,50244,4262,50637,0.085841,0.084168,1.988164,0.952898,0.340642,False
7,2,new_accounts,4184,50244,4165,50637,0.083274,0.082252,1.241934,0.588793,0.556000,False
8,3,add_payment_info,3697,70439,3623,70047,0.052485,0.051722,1.474630,0.643172,0.520112,False
9,3,add_shipping_info,5188,70439,5298,70047,0.073652,0.075635,-2.621211,-1.413727,0.157442,False


In [ ]:
# 5. RESULTS FORMATTING

results_df["conversion_rate_test"] = results_df["conversion_rate_test"].round(6)
results_df["conversion_rate_control"] = results_df["conversion_rate_control"].round(6)
results_df["metric_change"] = results_df["metric_change"].round(2)
results_df["z_stat"] = results_df["z_stat"].round(4)
results_df["p_value"] = results_df["p_value"].round(6)

results_df

,test_number,metric,numerator_event,denominator,numerator_control,denominator_control,conversion_rate_test,conversion_rate_control,metric_change,z_stat,p_value,significant
0,1,add_payment_info,2229,45193,1988,45362,0.049322,0.043825,12.54,3.9249,0.000087,True
1,1,add_shipping_info,3221,45193,3034,45362,0.071272,0.066884,6.56,2.6036,0.009226,True
2,1,begin_checkout,4021,45193,3784,45362,0.088974,0.083418,6.66,2.9788,0.002894,True
3,1,new_accounts,3681,45193,3823,45362,0.081451,0.084278,-3.35,-1.5429,0.122859,False
4,2,add_payment_info,2409,50244,2344,50637,0.047946,0.046290,3.58,1.2410,0.214608,False
5,2,add_shipping_info,3510,50244,3480,50637,0.069859,0.068724,1.65,0.7096,0.477979,False
6,2,begin_checkout,4313,50244,4262,50637,0.085841,0.084168,1.99,0.9529,0.340642,False
7,2,new_accounts,4184,50244,4165,50637,0.083274,0.082252,1.24,0.5888,0.556000,False
8,3,add_payment_info,3697,70439,3623,70047,0.052485,0.051722,1.47,0.6432,0.520112,False
9,3,add_shipping_info,5188,70439,5298,70047,0.073652,0.075635,-2.62,-1.4137,0.157442,False


In [ ]:
# Export Excel with explicit comma separator and UTF-8 encoding
results_df.to_excel("ab_test_results.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("ab_test_results.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[Link on Dashboard](https://public.tableau.com/app/profile/iryna.matviichuk/viz/ABTest_17822407079460/ExperimentOverview?publish=yes)

## Key Findings

- Test 1 showed statistically significant improvements in:
  - add_payment_info (+12.54%)
  - add_shipping_info (+6.56%)
  - begin_checkout (+6.66%)

- Test 2 showed no statistically significant changes.

- Test 3 showed a statistically significant decline in begin_checkout (-3.35%).

- Test 4 showed statistically significant declines in:
  - begin_checkout (-2.35%)
  - new_accounts (-3.36%)